In [4]:
import pandas as pd
from scipy import stats

# 1. Load both result files
df_nyt = pd.read_csv("../data/bot5_chronological_results.csv")
df_rew = pd.read_csv("../data/bot5_chronological_results_2.csv")

# 2. Keep only the FIRST instance of each target word
df_nyt_dedup = df_nyt.drop_duplicates(subset=['target'], keep='first').copy()
df_rew_dedup = df_rew.drop_duplicates(subset=['target'], keep='first').copy()

# 3. Merge deduplicated datasets
merged = df_nyt_dedup.merge(df_rew_dedup, on="target", suffixes=("_nyt", "_rew"))

def analyze_significance(df, label):
    diffs = df["guess_count_nyt"] - df["guess_count_rew"]
    non_zero_diffs = diffs[diffs != 0]
    
    if len(non_zero_diffs) > 0:
        t_stat, p_val_ttest = stats.ttest_rel(df["guess_count_nyt"], df["guess_count_rew"])
        stat_wilc, p_val_wilc = stats.wilcoxon(non_zero_diffs)
    else:
        p_val_ttest, p_val_wilc = 1.0, 1.0

    print("=" * 60)
    print(f"--- STATISTICAL SIGNIFICANCE: {label} ---")
    print("=" * 60)
    print(f"Total Unique Targets Evaluated: {len(df):,}")
    print(f"Net Guesses Saved:              {diffs.sum()}")
    print(f"Games Improved (+1+):           {(diffs > 0).sum()}")
    print(f"Games Worsened (-1+):           {(diffs < 0).sum()}")
    print(f"Games Identical:                {(diffs == 0).sum()}")
    print("-" * 60)
    print(f"Paired t-test p-value:          {p_val_ttest:.6f}")
    print(f"Wilcoxon p-value:               {p_val_wilc:.6f}")
    
    sig_status = "YES (Statistically Significant)" if p_val_ttest < 0.05 else "NO (Not Significant)"
    print(f"Significant at alpha = 0.05?    {sig_status}")
    print("=" * 60 + "\n")

# Run test on ALL unique target words
analyze_significance(merged, "ALL UNIQUE TARGETS")

# Run test on 2023 ONWARDS ONLY
merged_2023 = merged[merged["year_nyt"] >= 2023].copy()
analyze_significance(merged_2023, "2023 ONWARDS ONLY")

--- STATISTICAL SIGNIFICANCE: ALL UNIQUE TARGETS ---
Total Unique Targets Evaluated: 1,838
Net Guesses Saved:              40
Games Improved (+1+):           210
Games Worsened (-1+):           170
Games Identical:                1458
------------------------------------------------------------
Paired t-test p-value:          0.046554
Wilcoxon p-value:               0.046746
Significant at alpha = 0.05?    YES (Statistically Significant)

--- STATISTICAL SIGNIFICANCE: 2023 ONWARDS ONLY ---
Total Unique Targets Evaluated: 1,277
Net Guesses Saved:              37
Games Improved (+1+):           150
Games Worsened (-1+):           113
Games Identical:                1014
------------------------------------------------------------
Paired t-test p-value:          0.028903
Wilcoxon p-value:               0.029206
Significant at alpha = 0.05?    YES (Statistically Significant)



In [3]:
from pathlib import Path
import pandas as pd

# Load result CSVs
data_dir = Path("../data")  # Adjust if your data folder relative path differs
df_b5 = pd.read_csv(data_dir / "bot5_chronological_results_2.csv")
df_b6 = pd.read_csv(data_dir / "bot6_chronological_results.csv")

# Align Bot 5 with Bot 6's evaluation window using game_num
merged = pd.merge(
    df_b6,
    df_b5[["game_num", "guess_count"]],
    on="game_num",
    suffixes=("_b6", "_b5"),
)

# Subset masks
is_rep = merged["is_repeat"] == True
is_non_rep = merged["is_repeat"] == False

# Calculate metrics breakdown
summary_data = [
    {
        "Category": "Overall Era",
        "Games": len(merged),
        "Bot 5 Avg": merged["guess_count_b5"].mean(),
        "Bot 6 Avg": merged["guess_count_b6"].mean(),
    },
    {
        "Category": "Repeats Only",
        "Games": is_rep.sum(),
        "Bot 5 Avg": merged.loc[is_rep, "guess_count_b5"].mean(),
        "Bot 6 Avg": merged.loc[is_rep, "guess_count_b6"].mean(),
    },
    {
        "Category": "Non-Repeats Only",
        "Games": is_non_rep.sum(),
        "Bot 5 Avg": merged.loc[is_non_rep, "guess_count_b5"].mean(),
        "Bot 6 Avg": merged.loc[is_non_rep, "guess_count_b6"].mean(),
    },
]

df_comparison = pd.DataFrame(summary_data)
df_comparison["Diff (B6 - B5)"] = (
    df_comparison["Bot 6 Avg"] - df_comparison["Bot 5 Avg"]
)

# Display pretty formatted output
df_comparison.style.format(
    {
        "Bot 5 Avg": "{:.4f}",
        "Bot 6 Avg": "{:.4f}",
        "Diff (B6 - B5)": "{:+.4f}",
    }
)

,Category,Games,Bot 5 Avg,Bot 6 Avg,Diff (B6 - B5)
0,Overall Era,167,3.2934,3.1856,-0.1078
1,Repeats Only,18,4.6667,3.8889,-0.7778
2,Non-Repeats Only,149,3.1275,3.1007,-0.0268
